In [ ]:
import pandas as pd
import arviz as az
from matplotlib import pyplot as plt
import seaborn as sns

from emu_renewal.inputs import OUTPUTS_PATH

In [ ]:
job_path = OUTPUTS_PATH / "42523356"
countries = [p.parts[-1] for p in job_path.iterdir()]
country = countries[0]
country_path = job_path / country
analyses = [a.parts[-1] for a in country_path.iterdir()]

In [ ]:
collated_likes = {}
for a in analyses:
    analysis_path = country_path / a
    idata = az.from_netcdf(analysis_path / "idata_filtered.nc")
    total_like = idata["sample_stats"]["lp"].to_pandas().T
    all_likes = pd.concat([idata["log_likelihood"].to_dataframe().reset_index(), total_like.melt()["value"]], axis=1)
    all_likes = all_likes.rename(columns={"value": "total_ll"})
    collated_likes[a] = all_likes

In [ ]:
components = collated_likes["no_mob"].columns[-1: 2: -1]

In [ ]:
like_comps_dfs = pd.DataFrame(columns=pd.MultiIndex.from_product([collated_likes.keys(), collated_likes["no_mob"].columns[2:]]))
for a in collated_likes:
    for c in collated_likes[a].columns[2:]:
        like_comps_dfs[a, c] = collated_likes[a][c]
like_comps_dfs = like_comps_dfs.swaplevel(0, 1, axis=1)

In [ ]:
comp_fig, axes = plt.subplots(3, 2, figsize=[11, 14])
flat_axes = axes.ravel()
for t, targ in enumerate(set(like_comps_dfs.columns.get_level_values(0))):
    ax = sns.kdeplot(like_comps_dfs[targ], fill=True, ax=flat_axes[t])
    ax.set_title(targ)
    if t != 0:
        flat_axes[t].get_legend().remove()
comp_fig.suptitle(country, fontsize=15)
comp_fig.tight_layout()